In [ ]:
import json
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader,TextLoader,PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:
load_dotenv(override=True)

In [ ]:
cle_extraite = os.getenv("CUAD_KEY")

llm = ChatMistralAI(
    model="mistral-small-latest", 
    temperature=0.0,
    api_key=cle_extraite 
)

In [ ]:
os.getenv("CUAD_KEY")

# 1 - Loading the PDF , chuncking

In [ ]:
pdf_path = "DIABATE YOUSSOUF ? Data Scientist & AI Engineer.pdf"
pdf_loader = PyPDFLoader(pdf_path)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=20,
    encoding_name="gpt2"
)

In [ ]:
chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(chunks)

In [ ]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="QA_CV",
    persist_directory="./chroma_db"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
retrieved_chunks = retriever.invoke("What is the name of the person in the CV ?")

In [ ]:
retrieved_chunks

In [ ]:
print(retrieved_chunks[0].page_content)

In [ ]:
prompt = """You are a helpful assistant for answering questions about a CV. 
        Use the following retrieved chunks to answer the question. If you don't know the answer, say you don't know.
        <context>
        {context}
        </context>
        <question>
        {question}
        </question>
        """

## Retriever

In [ ]:
user_question = "What is the phone number of the person in the CV ?"
relevant_document_chunks = retriever.invoke(user_question)
context_list = [chunk.page_content for chunk in relevant_document_chunks]
context_for_query = "\n".join(context_list)

In [ ]:
print(context_for_query)

In [ ]:
prompt = prompt.format(context=context_for_query, question=user_question)

In [ ]:
print(prompt)

In [ ]:
resp = llm.invoke(prompt)

In [ ]:
print(resp.content)

In [ ]:
def RAG(query,llm=llm,prompt_template=prompt):
    relevant_document_chunks = retriever.invoke(query)
    context_list = [chunk.page_content for chunk in relevant_document_chunks]
    context_for_query = "\n".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    response = llm.invoke(prompt)
    return response.content
 

In [ ]:
print(RAG("What is the competences  of the person in the CV ?"))

In [ ]:
print(RAG("I'm angry, i want to eat"))

In [ ]:
print(RAG("Who are you ?"))

In [ ]:
print(RAG("Who are you ?"))

In [ ]:
print(RAG("What is her formation in Data Science ?"))